In [ ]:
import sqlite3
import pandas as pd

DB_PATH = '/Users/muna/Hana_research/data/db/Hana_Research.db'
OUTPUT_CSV = '/Users/muna/Hana_research/data/output/tolvaptan_lab_wide.csv'

ITEM_CODES = ['0063', '0074', '0087', '0093', '0099', '2550', '2551']

con = sqlite3.connect(DB_PATH)

In [ ]:
# --- 1. tolvaptan_study から Patient_ID と ind_date 取得 ---
tol = pd.read_sql(
    "SELECT Patient_ID, ind_date FROM tolvaptan_study",
    con,
    parse_dates=['ind_date']
)
print(f'tolvaptan_study: {len(tol)} rows')
tol.head()

In [ ]:
# --- 2. 対象 item_code の lab_clean 取得 ---
item_placeholder = ','.join(['?' for _ in ITEM_CODES])
lab = pd.read_sql(
    f"""
    SELECT Patient_ID, item_code, item_name, value_raw, sample_date
    FROM lab_clean
    WHERE item_code IN ({item_placeholder})
    """,
    con,
    params=ITEM_CODES,
    parse_dates=['sample_date']
)
print(f'lab_clean (filtered): {len(lab)} rows')
lab.head()

In [ ]:
# --- 3. 各患者ごとに「基準日以前1回」＋「翌日以降4回」を選択 ---

def pick_dates(patient_id, ind_date, lab_df):
    """Returns up to 5 sample_dates: 1 before/on ind_date + 4 after."""
    p_lab = lab_df[lab_df['Patient_ID'] == patient_id].copy()
    p_lab = p_lab.drop_duplicates(subset=['item_code', 'sample_date'])

    # 基準日以前：最も近い採血日（当日含む）
    before = (
        p_lab[p_lab['sample_date'] <= ind_date]['sample_date']
        .drop_duplicates()
        .sort_values(ascending=False)
    )
    pre_date = [before.iloc[0]] if len(before) > 0 else []

    # 翌日以降：昇順に最大4回分の採血日
    after = (
        p_lab[p_lab['sample_date'] > ind_date]['sample_date']
        .drop_duplicates()
        .sort_values()
    )
    post_dates = list(after.iloc[:4])

    return pre_date + post_dates  # max 5 dates


records = []

for _, row in tol.iterrows():
    pid = row['Patient_ID']
    ind = row['ind_date']
    selected_dates = pick_dates(pid, ind, lab)

    # 全 item_code × 全採血日のgrid
    for icode in ITEM_CODES:
        # item_name を取得（lab_clean にあれば）
        iname_series = lab[
            (lab['Patient_ID'] == pid) & (lab['item_code'] == icode)
        ]['item_name']
        iname = iname_series.iloc[0] if len(iname_series) > 0 else ''

        rec = {
            'Patient_ID': pid,
            'ind_date': ind.date(),
            'item_code': icode,
            'item_name': iname,
        }

        for i, dt in enumerate(selected_dates):
            label = f'value_T{i}' if i > 0 else 'value_pre'
            date_label = f'date_T{i}' if i > 0 else 'date_pre'
            # 該当日・同item_codeの値（複数あれば最初の1件）
            match = lab[
                (lab['Patient_ID'] == pid) &
                (lab['item_code'] == icode) &
                (lab['sample_date'] == dt)
            ]
            rec[date_label] = dt.date() if len(match) > 0 else None
            rec[label] = match['value_raw'].iloc[0] if len(match) > 0 else None

        records.append(rec)

result = pd.DataFrame(records)
print(f'Output: {len(result)} rows')
result.head(14)

In [ ]:
# --- 4. CSV 出力 ---
import os
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
result.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')  # Excel対応BOM付き
print(f'Saved: {OUTPUT_CSV}')

con.close()